In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q11/q11_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 3 ###
# select needed columns and compute TOTAL_COST on GPU
partsupp_filtered = partsupp[["PS_PARTKEY","PS_SUPPKEY","PS_SUPPLYCOST","PS_AVAILQTY"]]
partsupp_filtered["TOTAL_COST"] = \
    partsupp_filtered["PS_SUPPLYCOST"] * partsupp_filtered["PS_AVAILQTY"]

# select supplier keys
supplier_filtered = supplier[["S_SUPPKEY","S_NATIONKEY"]]

# join partsupp with supplier and keep only needed columns
ps_supp_merge = (
    partsupp_filtered
    .merge(
        supplier_filtered,
        left_on="PS_SUPPKEY",
        right_on="S_SUPPKEY",
        how="inner"
    )
    [["PS_PARTKEY","S_NATIONKEY","TOTAL_COST"]]
)

# filter nation to Germany and keep only key
nation_filtered = nation[nation["N_NAME"] == "GERMANY"][["N_NATIONKEY"]]

# join with filtered nation and keep only partkey and cost
ps_supp_n_merge = (
    ps_supp_merge
    .merge(
        nation_filtered,
        left_on="S_NATIONKEY",
        right_on="N_NATIONKEY",
        how="inner"
    )
    [["PS_PARTKEY","TOTAL_COST"]]
)

# compute threshold
sum_val = ps_supp_n_merge["TOTAL_COST"].sum() * 0.0001

# group on GPU, sum and rename without NamedAgg
total = (
    ps_supp_n_merge
    .groupby("PS_PARTKEY", sort=False)["TOTAL_COST"]
    .sum()
    .reset_index()
)
total.columns = ["PS_PARTKEY","VALUE"]

# filter and sort
total = total[total["VALUE"] > sum_val].sort_values("VALUE", ascending=False)